# Konsistenzprüfung mit DeepSeek R1 Zero

Dieses Notebook lädt die zuvor extrahierten Claims und KPIs, ruft DeepSeek R1 Zero über die OpenRouter-API auf und bewertet, ob die Aussagen durch die Kennzahlen gestützt oder widerlegt werden.

## Voraussetzungen
- Erfolgreich erzeugte CSV-Dateien aus den Notebooks `01_claim_extraction.ipynb` und `02_kpi_extraction.ipynb`.
- Ein gültiger OpenRouter API Key in der Umgebungsvariable `OPENROUTER_API_KEY`.
- Internetzugang zum Aufruf des Modells `deepseek/deepseek-r1-zero`.

In [ ]:
from pathlib import Path
import json
import ast
import pandas as pd
from collections import defaultdict
import os

import sys
PROJECT_ROOT = Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from greenwashing_pipeline.claim_extraction import ClaimCandidate
from greenwashing_pipeline.kpi_extraction import KPIRecord
from greenwashing_pipeline.llm_evaluator import ConsistencyEvaluator

## Pfade und Optionen einstellen
Mit `run_evaluation` kann der API-Aufruf testweise deaktiviert werden.

In [ ]:
claims_csv = Path("data/claims.csv")
kpis_csv = Path("data/kpis.csv")
output_json = Path("data/evaluations.json")
run_evaluation = True  # auf False setzen, um DeepSeek nicht aufzurufen

print(f"Claims-CSV: {claims_csv}")
print(f"KPI-CSV: {kpis_csv}")
print(f"Ergebnis-JSON: {output_json}")

## CSV-Dateien laden und in Python-Objekte umwandeln

In [ ]:
claims_df = pd.read_csv(claims_csv)
kpis_df = pd.read_csv(kpis_csv)

claims = [
    ClaimCandidate(
        text=row["text"],
        label=row["label"],
        score=float(row["score"]),
        document_id=row["document_id"],
        page_number=int(row["page"]),
    )
    for _, row in claims_df.iterrows()
]

def parse_metadata(value):
    if isinstance(value, dict):
        return value
    if pd.isna(value):
        return {}
    try:
        return ast.literal_eval(value)
    except (ValueError, SyntaxError):
        return {}

kpirecords = []
for _, row in kpis_df.iterrows():
    metadata = parse_metadata(row.get("metadata"))
    kpirecords.append(
        KPIRecord(
            kpi_type=row["kpi_type"],
            value=float(row["value"]),
            unit=row.get("unit") if not pd.isna(row.get("unit")) else None,
            year=int(row["year"]) if not pd.isna(row.get("year")) else None,
            source_text=row.get("source_text", ""),
            document_id=row["document_id"],
            page_number=int(row["page_number"]),
            metadata=metadata,
        )
    )

print(f"Claims geladen: {len(claims)}")
print(f"KPIs geladen: {len(kpirecords)}")

## KPIs je Dokument bündeln

In [ ]:
kpis_by_doc = defaultdict(list)
for kpi in kpirecords:
    kpis_by_doc[kpi.document_id].append(kpi)

{doc: len(items) for doc, items in kpis_by_doc.items()}

## LLM-Auswertung anstoßen

In [ ]:
if run_evaluation:
    if not os.getenv("OPENROUTER_API_KEY"):
        raise RuntimeError("OPENROUTER_API_KEY ist nicht gesetzt. Bitte vor dem Ausführen hinterlegen oder `run_evaluation = False` verwenden.")
    evaluator = ConsistencyEvaluator()
    evaluations = []
    for claim in claims:
        doc_kpis = kpis_by_doc.get(claim.document_id, [])
        result = evaluator.evaluate(claim, doc_kpis)
        evaluations.append(result)
else:
    print("Evaluation übersprungen – es werden Dummy-Ergebnisse erzeugt.")
    evaluations = []

len(evaluations)

## Ergebnisse speichern
Die DeepSeek-Antworten werden als JSON-Datei abgelegt und können anschließend analysiert oder visualisiert werden.

In [ ]:
payload = [evaluation.to_dict() for evaluation in evaluations]
output_json.parent.mkdir(parents=True, exist_ok=True)
output_json.write_text(json.dumps(payload, ensure_ascii=False, indent=2))
print(f"Gespeicherte Bewertungen: {len(payload)}")